# Clasificación de Imágenes desde Cero (Cats vs Dogs) - Versión Optimizada
**Computación Emergente - UJAP**  
**Referencia original:** [Keras Examples - Image classification from scratch](https://keras.io/examples/vision/image_classification_from_scratch/)

Este cuaderno implementa un pipeline integral de Deep Learning para clasificación binaria entrenando una red convolucional (Mini-Xception) desde cero con Keras 3 y TensorFlow.

**Mejoras incorporadas:**
- **Aceleración por hardware:** Crecimiento dinámico de VRAM y Precisión Mixta FP16 para aprovechar Tensor Cores.
- **Saneamiento robusto:** Filtrado exhaustivo de imágenes corruptas o truncadas con Pillow (`verify` + `load`).
- **Partición en 3 vías:** Entrenamiento (80%), Validación (10%) y Prueba independiente (10%).
- **Data Augmentation enriquecido:** Giros horizontales, rotaciones, zooms y variaciones de contraste.
- **Métricas integrales:** Exactitud (Acc), Precisión, Recall y AUC.
- **Callbacks avanzados:** `ModelCheckpoint` (solo el mejor según `val_loss`), `EarlyStopping` y `ReduceLROnPlateau`.
- **Evaluación y Explicabilidad:** Curvas de aprendizaje, evaluación sobre Test Set y Matriz de Confusión.


## 1. Importación de dependencias y configuración de Hardware (GPU / Precisión Mixta)


In [ ]:
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

os.environ.setdefault("KERAS_BACKEND", "tensorflow")
import keras
from keras import layers
import tensorflow as tf
from tensorflow import data as tf_data

print("Keras version:", keras.__version__)
print("TensorFlow version:", tf.__version__)

# Configuración de GPU y Precisión Mixta FP16
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU(s) detectada(s):", [g.name for g in gpus])
        keras.mixed_precision.set_global_policy("mixed_float16")
        print("Precisión mixta ('mixed_float16') activada con éxito.")
    except Exception as e:
        print("Configuración GPU:", e)
else:
    print("Sin GPU dedicada detectada. Ejecutando en CPU.")


## 2. Descargar y preparar el dataset de Kaggle (Cats vs Dogs)


In [ ]:
import urllib.request
import zipfile

data_url = "https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip"
zip_name = "kagglecatsanddogs_5340.zip"
data_dir = "PetImages"

if not os.path.exists(data_dir):
    if not os.path.exists(zip_name):
        print("Descargando dataset...")
        urllib.request.urlretrieve(data_url, zip_name)
    print("Extrayendo archivo...")
    with zipfile.ZipFile(zip_name, "r") as zip_ref:
        zip_ref.extractall(".")
    print("Extracción completada.")
else:
    print(f"El directorio '{data_dir}' ya existe.")


### 2.1 Filtrar imágenes corruptas con Pillow (PIL)
Utilizamos Pillow para verificar tanto las cabeceras como la decodificación completa de píxeles (`verify` + `load`), asegurando que ninguna imagen truncada afecte el pipeline de entrenamiento.


In [ ]:
print("Verificando integridad de imágenes con Pillow...")
num_skipped = 0
for folder_name in ("Cat", "Dog"):
    folder_path = os.path.join(data_dir, folder_name)
    if not os.path.exists(folder_path):
        continue
    for fname in os.listdir(folder_path):
        fpath = os.path.join(folder_path, fname)
        is_corrupt = False
        try:
            with Image.open(fpath) as img:
                img.verify()
            with Image.open(fpath) as img:
                img.load()
        except Exception:
            is_corrupt = True

        if is_corrupt:
            num_skipped += 1
            try:
                os.remove(fpath)
            except OSError:
                pass

print(f"Se eliminaron {num_skipped} imágenes corruptas o truncadas.")


## 3. Generar particiones de datos: Train (80%), Val (10%) y Test (10%)
Dividimos el dataset de forma reproducible:
- **Entrenamiento (80%):** Para ajustar los parámetros de la red.
- **Validación (10%):** Para monitoreo de sobreajuste y activación de callbacks (`EarlyStopping`, `ReduceLROnPlateau`).
- **Prueba / Test (10%):** Conjunto ciego para evaluación final objetiva.


In [ ]:
image_size = (180, 180)
batch_size = 128

train_ds, val_test_full_ds = keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="both",
    seed=1337,
    image_size=image_size,
    batch_size=batch_size,
)

val_test_batches = tf.data.experimental.cardinality(val_test_full_ds).numpy()
test_batches = val_test_batches // 2
val_batches = val_test_batches - test_batches

val_ds = val_test_full_ds.skip(test_batches)
test_ds = val_test_full_ds.take(test_batches)

print(f"Lotes de Entrenamiento: {tf.data.experimental.cardinality(train_ds).numpy()}")
print(f"Lotes de Validación:    {val_batches}")
print(f"Lotes de Prueba (Test): {test_batches}")


## 4. Visualización de los datos de muestra


In [ ]:
plt.figure(figsize=(9, 9))
for images, labels in train_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(np.array(images[i]).astype("uint8"))
        label_text = "Perro (1)" if int(labels[i]) == 1 else "Gato (0)"
        plt.title(label_text, fontsize=11, fontweight="bold")
        plt.axis("off")
plt.tight_layout()
plt.show()


## 5. Aumento de Datos (*Data Augmentation*) y Optimización del Pipeline
Enriquecemos el dataset con transformaciones de giro horizontal, rotación aleatoria, zoom y contraste para prevenir sobreajuste.


In [ ]:
data_augmentation_layers = [
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
]

def data_augmentation(images):
    for layer in data_augmentation_layers:
        images = layer(images)
    return images

# Optimizar el pipeline de datos con tf.data y prefetching
train_ds = train_ds.map(
    lambda img, label: (data_augmentation(img), label),
    num_parallel_calls=tf_data.AUTOTUNE,
)
train_ds = train_ds.prefetch(tf_data.AUTOTUNE)
val_ds = val_ds.prefetch(tf_data.AUTOTUNE)
test_ds = test_ds.prefetch(tf_data.AUTOTUNE)


## 6. Construir la Arquitectura de Red (Mini-Xception)
Implementa convoluciones separables en profundidad (`SeparableConv2D`), conexiones residuales de atajo (*skip connections*), `BatchNormalization` y `Dropout`. La capa densa de salida se fija explícitamente en `float32` para garantizar estabilidad numérica al utilizar precisión mixta FP16.


In [ ]:
def make_model(input_shape, num_classes):
    inputs = keras.Input(shape=input_shape)

    # Bloque inicial
    x = layers.Rescaling(1.0 / 255)(inputs)
    x = layers.Conv2D(128, 3, strides=2, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    previous_block_activation = x

    for size in [256, 512, 728]:
        x = layers.Activation("relu")(x)
        x = layers.SeparableConv2D(size, 3, padding="same")(x)
        x = layers.BatchNormalization()(x)

        x = layers.Activation("relu")(x)
        x = layers.SeparableConv2D(size, 3, padding="same")(x)
        x = layers.BatchNormalization()(x)

        x = layers.MaxPooling2D(3, strides=2, padding="same")(x)

        residual = layers.Conv2D(size, 1, strides=2, padding="same")(
            previous_block_activation
        )
        x = layers.add([x, residual])
        previous_block_activation = x

    x = layers.SeparableConv2D(1024, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    x = layers.GlobalAveragePooling2D()(x)
    units = 1 if num_classes == 2 else num_classes

    x = layers.Dropout(0.25)(x)
    outputs = layers.Dense(units, activation=None, dtype="float32")(x)
    return keras.Model(inputs, outputs, name="mini_xception")

model = make_model(input_shape=image_size + (3,), num_classes=2)
model.summary()


## 7. Compilación y Callbacks Inteligentes
Compilamos con métricas completas (`BinaryAccuracy`, `Precision`, `Recall`, `AUC`) y configuramos:
- `ModelCheckpoint` guardando únicamente el mejor modelo según `val_loss`.
- `EarlyStopping` para cortar el entrenamiento si no hay mejora tras 5 épocas consecutivas.
- `ReduceLROnPlateau` para refinar el ajuste cuando la pérdida se estabiliza.


In [ ]:
epochs = 25

callbacks = [
    keras.callbacks.ModelCheckpoint(
        "best_model.keras",
        monitor="val_loss",
        save_best_only=True,
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1,
    ),
]

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=3e-4),
    loss=keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=[
        keras.metrics.BinaryAccuracy(name="acc"),
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall"),
        keras.metrics.AUC(name="auc"),
    ],
)

history = model.fit(
    train_ds,
    epochs=epochs,
    callbacks=callbacks,
    validation_data=val_ds,
)

model.save("final_model.keras")
print("Modelo persistido en final_model.keras")


## 8. Gráficas de Rendimiento y Curvas de Aprendizaje


In [ ]:
epochs_range = range(1, len(history.history["loss"]) + 1)
fig, axs = plt.subplots(2, 2, figsize=(14, 10))

# Pérdida
axs[0, 0].plot(epochs_range, history.history["loss"], label="Train Loss", marker="o")
axs[0, 0].plot(epochs_range, history.history["val_loss"], label="Val Loss", marker="o")
axs[0, 0].set_title("Pérdida (Binary Crossentropy)", fontweight="bold")
axs[0, 0].set_xlabel("Épocas")
axs[0, 0].grid(True, linestyle="--", alpha=0.6)
axs[0, 0].legend()

# Exactitud
axs[0, 1].plot(epochs_range, history.history["acc"], label="Train Acc", marker="o")
axs[0, 1].plot(epochs_range, history.history["val_acc"], label="Val Acc", marker="o")
axs[0, 1].set_title("Exactitud (Accuracy)", fontweight="bold")
axs[0, 1].set_xlabel("Épocas")
axs[0, 1].grid(True, linestyle="--", alpha=0.6)
axs[0, 1].legend()

# AUC
if "auc" in history.history:
    axs[1, 0].plot(epochs_range, history.history["auc"], label="Train AUC", marker="o")
    axs[1, 0].plot(epochs_range, history.history["val_auc"], label="Val AUC", marker="o")
    axs[1, 0].set_title("Área Bajo la Curva ROC (AUC)", fontweight="bold")
    axs[1, 0].set_xlabel("Épocas")
    axs[1, 0].grid(True, linestyle="--", alpha=0.6)
    axs[1, 0].legend()

# Tasa de Aprendizaje o Precision/Recall
lr_key = next((k for k in ("lr", "learning_rate") if k in history.history), None)
if lr_key:
    axs[1, 1].plot(epochs_range, history.history[lr_key], label="Learning Rate", color="purple", marker="o")
    axs[1, 1].set_title("Learning Rate Dinámico", fontweight="bold")
    axs[1, 1].set_yscale("log")
    axs[1, 1].set_xlabel("Épocas")
    axs[1, 1].grid(True, linestyle="--", alpha=0.6)
    axs[1, 1].legend()
elif "val_precision" in history.history:
    axs[1, 1].plot(epochs_range, history.history["val_precision"], label="Val Precision", marker="o")
    axs[1, 1].plot(epochs_range, history.history["val_recall"], label="Val Recall", marker="o")
    axs[1, 1].set_title("Validación: Precision vs Recall", fontweight="bold")
    axs[1, 1].set_xlabel("Épocas")
    axs[1, 1].grid(True, linestyle="--", alpha=0.6)
    axs[1, 1].legend()

plt.tight_layout()
plt.show()


## 9. Evaluación en el Conjunto de Prueba (Test Set) y Matriz de Confusión
Evaluamos el modelo óptimo (`best_model.keras`) sobre datos ciegos que jamás observó en entrenamiento ni validación.


In [ ]:
best_model = keras.models.load_model("best_model.keras") if os.path.exists("best_model.keras") else model

print("Resultados en Test Set:")
test_results = best_model.evaluate(test_ds, return_dict=True)
for k, v in test_results.items():
    print(f"  {k.upper()}: {v:.4f}")

# Calcular predicciones para la Matriz de Confusión
y_true_list = []
y_pred_list = []

for images, labels in test_ds:
    preds = best_model.predict(images, verbose=0)
    probs = keras.ops.sigmoid(preds)
    probs_np = keras.ops.convert_to_numpy(probs)
    binary_preds = (probs_np > 0.5).astype(int).flatten()

    y_pred_list.extend(binary_preds)
    y_true_list.extend(keras.ops.convert_to_numpy(labels).flatten())

y_true_arr = np.array(y_true_list, dtype=int)
y_pred_arr = np.array(y_pred_list, dtype=int)

tp = np.sum((y_true_arr == 1) & (y_pred_arr == 1))
tn = np.sum((y_true_arr == 0) & (y_pred_arr == 0))
fp = np.sum((y_true_arr == 0) & (y_pred_arr == 1))
fn = np.sum((y_true_arr == 1) & (y_pred_arr == 0))

cm = np.array([[tn, fp], [fn, tp]])

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap=plt.cm.Blues)
ax.figure.colorbar(im, ax=ax)
ax.set(
    xticks=[0, 1],
    yticks=[0, 1],
    xticklabels=["Gato", "Perro"],
    yticklabels=["Gato", "Perro"],
    title="Matriz de Confusión (Test Set)",
    ylabel="Etiqueta Real",
    xlabel="Etiqueta Predicha",
)
thresh = cm.max() / 2.0 if cm.max() > 0 else 1.0
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black",
                fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

precision_val = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall_val = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1_val = (2 * precision_val * recall_val / (precision_val + recall_val)) if (precision_val + recall_val) > 0 else 0.0
print(f"F1-Score (Perro): {f1_val:.4f}")


## 10. Inferencia sobre una imagen individual


In [ ]:
sample_image = "PetImages/Cat/6779.jpg"

if os.path.exists(sample_image):
    img = keras.utils.load_img(sample_image, target_size=image_size)
    plt.imshow(img)
    plt.axis("off")
    plt.title("Imagen de Prueba", fontweight="bold")
    plt.show()

    img_array = keras.utils.img_to_array(img)
    img_array = keras.ops.expand_dims(img_array, 0)

    predictions = best_model.predict(img_array)
    score = float(keras.ops.sigmoid(predictions[0][0]))
    print(f"Predicción: {100 * (1 - score):.2f}% Gato, {100 * score:.2f}% Perro.")
else:
    print(f"No se encontró la imagen {sample_image}")
